[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/08_tokens_multimodales_audio/08_tokens_multimodales_audio.ipynb)

# Módulo 08 · Tokens multimodales y de audio: parches ViT, VQ-VAE, FSQ, RVQ y códecs neuronales

| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración estimada** | ≈ 6 h (lección ≈ 3.5 h + proyecto ≈ 2.5 h) |
| **GPU recomendada** | `FAST_DEV_RUN=True`: **CPU** (≈ 6-10 min en total; MNIST y voz real FSDD, modelos de < 1M parámetros). Modo completo: **T4** (≈ 3-5 unidades de Colab, estimación) |
| **Prerrequisitos** | 02 (BPE como compresión), 04 (coste de contexto por idioma), 06 (bits por byte, parches); álgebra lineal y PyTorch (convoluciones) |
| **Madurez del tema** | 🟢 parches ViT y códecs de audio tipo EnCodec · 🟡 VQ/FSQ como interfaz con LLM, Mimi, tokenizers de vídeo · 🔴 tokenizers 1D y modelos «continuos» unificados |

> **Contexto Políglota Labs.** El producto va a indexar pronto **PDFs con diagramas** y **grabaciones de soporte** (módulo 16 construye el retrieval multimodal). Antes de que alguien proponga «meterlo todo en el
> LLM», tu CTO quiere saber **qué es un token de imagen o de audio, cuántos hacen falta y cuánto cuesta en contexto**. En esta lección los construyes: un VQ-VAE/FSQ/RVQ desde cero sobre imágenes pequeñas, un
> códec de audio de juguete, y un LM sobre tokens de imagen (un «mini-Chameleon»). Todo diminuto y **didáctico**: no compite con EnCodec ni con VQGAN.

## 1. Objetivos de aprendizaje

1. **Distinguir** tokens **continuos** (parches ViT) de **discretos** (códigos de un codebook) y saber cuándo se usa cada uno.
2. **Implementar** `patchify` y demostrar que la *patch embedding* de ViT es una convolución con `stride = patch`.
3. **Derivar e implementar** VQ-VAE (distancia mínima, pérdida de codebook y *commitment*, *straight-through*), **medir** PSNR, bits por imagen y **perplejidad de uso** del codebook, y reproducir el **colapso del codebook**.
4. **Implementar** FSQ (cuantización escalar finita) y RVQ (cuantización residual con *quantizer dropout*) y compararlos con VQ.
5. **Entrenar** un LM sobre tokens de imagen y generar dígitos (la idea de Chameleon/VQGAN-transformer a escala de juguete).
6. **Construir** un códec de audio neuronal de juguete (encoder conv → RVQ → decoder conv) y **medir** tasa (kbps), SNR y distancia log-espectral frente a μ-law.
7. **Calcular** tasas de tokens por segundo/imagen/vídeo y su **coste en contexto** (EnCodec, Mimi, Chameleon, Cosmos…).
8. **Etiquetar** la madurez de cada técnica (🟢/🟡/🔴) y **conectar** con embeddings multimodales (módulo 16) sin adelantarlo.

## 2. 🔁 Conexión con módulos anteriores

<details>
<summary>1 · (Módulo 02) BPE comprime texto fusionando pares frecuentes. ¿En qué se parece y en qué se diferencia VQ de BPE?</summary>

Ambos aprenden un **diccionario** de K entradas para representar los datos con menos símbolos. BPE es *discreto y exacto* (sin pérdida; el diccionario son cadenas de bytes); VQ es **con pérdida** (cada vector se aproxima por su centroide más cercano) y se entrena por gradiente junto con un decodificador. Ambos se evalúan con una relación tasa-distorsión: en BPE, bytes por token; en VQ, bits por token frente a error de reconstrucción.

</details>

<details>
<summary>2 · (Módulo 04/06) ¿Qué mide «bits por byte» y cómo se traduce a imágenes y audio?</summary>

Mide la información que cuesta cada byte original. Para imágenes se usa **bits por píxel (bpp)** y para audio **kbps**; en los tres casos es una **tasa** que se compara a igual distorsión.

</details>

<details>
<summary>3 · (Módulo 04) ¿Por qué importa el número de tokens que genera cada modalidad para un LLM?</summary>

El contexto de un transformer es finito y el coste de atención crece con $N^2$: una imagen de 1 024 tokens o un minuto de audio con 36 000 tokens consumen mucho más contexto que un párrafo de texto. La «tarifa por idioma» del módulo 04 es el mismo fenómeno con otra modalidad.

</details>

## 3. Intuición primero: tokenizar = cuantizar

Un LLM consume **secuencias**. Imágenes, audio y vídeo son señales continuas y enormes (una foto de 512x512 tiene 262 144 píxeles; un minuto de audio a 24 kHz tiene 1.44 millones de muestras). *Tokenizar* una modalidad significa **comprimirla en una secuencia corta de unidades** que el modelo sepa manejar. Hay dos grandes familias:

| Familia | Qué es un «token» | Ejemplos | Ventaja | Coste |
|---|---|---|---|---|
| **Continua** | un **vector** (p. ej. un parche 16x16 proyectado a 768 números) | ViT, CLIP/SigLIP, encoders de VLM | sin pérdida de cuantización; se conecta con embeddings (módulo 16) | no se puede predecir con cross-entropy sobre un vocabulario |
| **Discreta** | un **id** de un codebook de tamaño $K$ (o de varios, en RVQ) | VQ-VAE, VQGAN, FSQ, EnCodec, Mimi | encaja con «predecir el siguiente token» y con un vocabulario unificado texto+imagen+audio | **pérdida** por cuantización y problemas de entrenamiento (colapso del codebook) |

* **Analogía de compresión de datos.** Es un **códec con pérdida**: el encoder hace de *transformada*, el cuantizador fija la **tasa** y el decoder reconstruye. Todo se evalúa con una curva **tasa-distorsión**.
* **Analogía de recsys/IR.** La cuantización vectorial residual es la hermana del **Product Quantization** de FAISS (módulo 14) y de los **semantic IDs** de recomendación generativa (Rajput et al., 2023, arXiv:2305.05065), donde un RQ-VAE convierte cada ítem en una pequeña secuencia de códigos. Aquí la usamos para reconstruir señales; allí, para **indexar**.

> ### 🎨 GEMINI-IMAGEN · ID: `M08-IMG-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **Mapa de tokens por modalidad**
>
> **Qué muestra:** Muestra cómo cada modalidad se convierte en secuencia (parches continuos vs códigos discretos) y las tasas de tokens típicas.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Cuatro filas: texto con BPE, imagen con parches ViT, imagen con VQ discreto y audio con códec RVQ, con sus tasas de tokens
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, estilo infografía de paper (sans-serif legible, flechas claras, sin ilustraciones decorativas ni texto inventado). Paleta: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Formato 16:9.
> Título: "De señales continuas a secuencias de tokens".
> Cuatro filas, de arriba a abajo; en cada fila, a la izquierda el icono esquemático de la señal (un rectángulo con trazos simples, sin dibujos realistas), en el centro un bloque con el método y a la derecha una secuencia de cajitas (los tokens) con la tasa.
> Fila 1 "Texto": rectángulo con líneas de texto → bloque azul "Tokenizer BPE (módulos 02-04)" → cajitas con ids "412 · 87 · 3051 · 9" → "~1.3 tokens por palabra (orden de magnitud)".
> Fila 2 "Imagen (continuo)": cuadrícula 14x14 sobre un cuadrado → bloque azul "Parches ViT 16x16 + proyección lineal" → cajitas con vectores (cada una con la etiqueta "vector de 768 números") → "196 tokens para una imagen de 224x224".
> Fila 3 "Imagen (discreto)": cuadrado → bloque naranja "Encoder → cuantizador VQ/FSQ (codebook)" → cajitas con ids "5021 · 17 · 8190 · 733" → "1 024 tokens (codebook de 8192) para 512x512 en Chameleon".
> Fila 4 "Audio (discreto)": onda sinusoidal → bloque verde "Códec neuronal: encoder → RVQ (8 niveles) → decoder" → una matriz de 8 filas x 4 columnas de cajitas de ids (columnas = frames) → "EnCodec 24 kHz a 6 kbps: 75 frames/s x 8 niveles = 600 tokens/s; Mimi: 12.5 frames/s x 8 = 100 tokens/s".
> Una barra vertical a la derecha etiquetada "coste de contexto" con una flecha creciente de arriba a abajo.
> Evita: ilustraciones decorativas, texto ilegible, números distintos de los indicados, flechas ambiguas, logotipos, sombras 3D.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M08-IMG-01.png` en la carpeta del módulo y reemplaza esta celda por: `![Cuatro filas: texto con BPE, imagen con parches ViT, imagen con VQ discreto y audio con códec RVQ, con sus tasas de tokens](assets/M08-IMG-01.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> Texto   -> BPE                         -> [412][87][3051]...        ~1.3 tokens/palabra
> Imagen  -> parches ViT 16x16 + Linear  -> [v768][v768]...           196 tokens (224x224)         (continuo)
> Imagen  -> encoder + VQ/FSQ            -> [5021][17][8190]...       1024 tokens (512x512, K=8192) (discreto)
> Audio   -> encoder + RVQ x8 + decoder  -> 8 filas x frames de ids   75 Hz x 8 = 600 tok/s (EnCodec 6 kbps); Mimi: 12.5 Hz x 8 = 100 tok/s
> ```

> **👀 Qué debes observar**
>
> Dos preguntas ordenan el módulo: **(1) ¿discreto o continuo?** (¿necesito *predecir* el token o solo *leerlo*?) y **(2) ¿cuántos tokens por segundo/imagen?** (el coste de contexto). La tabla de tasas que calculamos abajo te deja ver que un minuto de audio puede pesar **más que una novela corta** en tokens.

In [ ]:
!pip install -q torch torchvision torchaudio numpy matplotlib scikit-learn scipy transformers

In [ ]:
import os, sys, time, math, random, warnings
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
FAST_DEV_RUN = True      # True: CPU, minutos. False: GPU (T4/L4), escala completa
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", min(4, os.cpu_count() or 1))))
print("device:", device, "| FAST_DEV_RUN:", FAST_DEV_RUN, "| torch", torch.__version__)

import re, io
import matplotlib.pyplot as plt
import torch.nn as nn, torch.nn.functional as F
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False})
AZUL, NARANJA, VERDE, GRIS = "#1F6FEB", "#F28E2B", "#2CA02C", "#6B7280"

Datos de imagen: **MNIST real** vía `torchvision` (si no hay red, figuras sintéticas 28x28) y dos **fotos reales** empaquetadas con scikit-learn.

In [ ]:
from PIL import Image, ImageDraw

def mnist_sintetico(n, seed=SEED):
    """Respaldo sin red: figuras geométricas 28x28 (círculo, cuadrado, cruz, triángulo) con posición/tamaño aleatorios."""
    r = np.random.RandomState(seed); X = np.zeros((n, 1, 28, 28), np.float32); y = r.randint(0, 4, n)
    for i in range(n):
        im = Image.new("L", (28, 28)); d = ImageDraw.Draw(im); c = r.randint(9, 19, 2); s = r.randint(4, 9)
        box = [c[0] - s, c[1] - s, c[0] + s, c[1] + s]
        if y[i] == 0: d.ellipse(box, outline=255, width=2)
        elif y[i] == 1: d.rectangle(box, outline=255, width=2)
        elif y[i] == 2: d.line([c[0] - s, c[1], c[0] + s, c[1]], fill=255, width=2); d.line([c[0], c[1] - s, c[0], c[1] + s], fill=255, width=2)
        else: d.polygon([(c[0], c[1] - s), (c[0] - s, c[1] + s), (c[0] + s, c[1] + s)], outline=255)
        X[i, 0] = np.asarray(im, np.float32) / 255
    return torch.from_numpy(X), torch.from_numpy(y)

def carga_imagenes(n):
    try:
        from torchvision import datasets
        ds = datasets.MNIST("./data", train=True, download=True)
        return ds.data[:n].float().div(255).unsqueeze(1), ds.targets[:n], "MNIST real"
    except Exception as e:
        print("[aviso] MNIST no disponible (", type(e).__name__, ") -> figuras SINTÉTICAS 28x28")
        X, y = mnist_sintetico(n); return X, y, "sintético (4 clases)"

N_IMG = 6000 if FAST_DEV_RUN else 60000
X_ALL, Y_ALL, FUENTE_IMG = carga_imagenes(N_IMG + 1000)
X_TR, X_TE, Y_TR, Y_TE = X_ALL[:N_IMG].to(device), X_ALL[N_IMG:].to(device), Y_ALL[:N_IMG].to(device), Y_ALL[N_IMG:].to(device)
N_CLASES = int(Y_ALL.max()) + 1
print(f"imágenes: {FUENTE_IMG} | train={tuple(X_TR.shape)} test={tuple(X_TE.shape)} | {N_CLASES} clases")

def foto(nombre="china.jpg"):
    """Foto real que viene EMPAQUETADA con scikit-learn (sin descargas); respaldo sintético si no está."""
    try:
        from sklearn.datasets import load_sample_image
        return Image.fromarray(load_sample_image(nombre)), "foto real (scikit-learn)"
    except Exception:
        r = np.random.RandomState(0); yy, xx = np.mgrid[0:427, 0:640]
        im = np.stack([128 + 100 * np.sin(xx / 40 + k) * np.cos(yy / (30 + 10 * k)) + r.randn(427, 640) * 6 for k in range(3)], -1)
        return Image.fromarray(np.clip(im, 0, 255).astype(np.uint8)), "imagen sintética"

## 4. Mapa de madurez (a oct-2026)

| Técnica | Madurez | Quién la usa / respaldo público | Fuente |
|---|---|---|---|
| Parches ViT (tokens continuos) | 🟢 | Base de los encoders visuales de CLIP/SigLIP y de VLMs abiertos que los conectan a un LLM (p. ej. LLaVA) | Dosovitskiy et al. 2020 (arXiv:2010.11929); Liu et al. 2023 (arXiv:2304.08485) |
| VQ-VAE | 🟢 (técnica) / 🟡 (como interfaz LLM) | Base de DALL·E 1, VQGAN y de los tokenizers de imagen discretos | van den Oord et al. 2017 (arXiv:1711.00937) |
| VQGAN (+ transformer *prior*) | 🟢 | CompVis; pérdida perceptual + adversarial sobre VQ | Esser et al. 2020 (arXiv:2012.09841) |
| **FSQ** | 🟡 | Tokenizers *discretos* de Cosmos (NVIDIA) con niveles (8,8,8,5,5,5) = 64 000 | Mentzer et al. 2023 (arXiv:2309.15505); Cosmos (arXiv:2501.03575) |
| LFQ / MAGVIT-v2 | 🟡 | Google/CMU; ICLR 2024; tokenizer de imagen+vídeo con vocabulario común | Yu et al. 2023 (arXiv:2310.05737) |
| Tokens de imagen en un LLM (early fusion): Chameleon, Emu3 | 🟡 | Meta (Chameleon, 2024); BAAI (Emu3, 2024) | arXiv:2405.09818; arXiv:2409.18869 |
| Híbridos continuos+LM: Transfusion, MAR/Fluid (sin cuantización vectorial) | 🔴/🟡 | Meta (Transfusion, hasta 7B); resultados en investigación | arXiv:2408.11039; arXiv:2406.11838; arXiv:2410.13863 |
| Tokenizers 1D de imagen (TiTok, FlexTok) | 🔴 | ByteDance/TUM (32 tokens por imagen); EPFL/Apple | arXiv:2406.07550; arXiv:2502.13967 |
| SoundStream, **EnCodec**, DAC (RVQ) | 🟢 | EnCodec es el tokenizer de MusicGen (50 Hz, 32 kHz); DAC: 44.1 kHz a 8 kbps (~90x) | arXiv:2107.03312; 2210.13438; 2306.06546; 2306.05284 |
| **Mimi** (tokens semántico-acústicos, 12.5 Hz) | 🟡 | Kyutai (Moshi): conversación en tiempo real | arXiv:2410.00037 |
| Tokenizers de vídeo 3D (Cosmos, MAGVIT-v2) | 🟡 | NVIDIA (pesos abiertos), Google | arXiv:2501.03575; arXiv:2310.05737 |

> Los modelos cerrados de frontera (multimodales nativos) **no documentan** sus tokenizers: lo que sabemos viene de papers abiertos.

## 5. Presupuesto de tokens: ¿cuánto contexto cuesta cada modalidad?

Antes de implementar nada, una **calculadora**. Cada fila usa una cifra de un paper o la fórmula de la arquitectura (marcamos con `~` lo aproximado).

In [ ]:
# Tasas de tokens por modalidad. Cada fila = (descripción, tokens, fuente/fórmula). Todo es aritmética sobre cifras de los papers
# o de la arquitectura; las aproximaciones están marcadas con "~".
def tokens_vit(res, p): return (res // p) ** 2
def tokens_video(frames, h, w, ct, cs): return -(-frames // ct) * (h // cs) * (w // cs)          # ~ (T/ct)·(H/cs)·(W/cs)
def tokens_audio(seg, hz, nq): return seg * hz * nq

FILAS = [
    ("Texto: 1 min de habla transcrita (~150 palabras)",           int(150 * 1.3),                     "~1.3 tokens/palabra (orden de magnitud)"),
    ("Imagen ViT-B/16, 224²",                                      tokens_vit(224, 16),                "(224/16)²"),
    ("Imagen ViT-L/14, 336²",                                      tokens_vit(336, 14),                "(336/14)²"),
    ("Imagen Chameleon, 512² (VQ, codebook 8192)",                 1024,                               "Chameleon Team 2024: 512x512 -> 1024 tokens"),
    ("Imagen TiTok (tokenizer 1D)",                                32,                                 "Yu et al. 2024: 'An Image is Worth 32 Tokens'"),
    ("Imagen Cosmos DI16x16, 512² (FSQ 64 000)",                   tokens_vit(512, 16),                "512/16 en cada eje"),
    ("Audio 1 min EnCodec 24 kHz @1.5 kbps (2 codebooks)",         tokens_audio(60, 75, 2),            "75 Hz x 2"),
    ("Audio 1 min EnCodec 24 kHz @6 kbps (8 codebooks)",           tokens_audio(60, 75, 8),            "75 Hz x 8"),
    ("Audio 1 min Mimi (12.5 Hz x 8 codebooks, aplanado)",         tokens_audio(60, 12.5, 8),          "Mimi: 12.5 Hz, 8 x 2048, 1.1 kbps"),
    ("Vídeo 5 s, 24 fps, 720p, Cosmos DV8x16x16 (~)",              tokens_video(120, 720, 1280, 8, 16), "~ (T/8)·(H/16)·(W/16)"),
]
df_tok = [(n, int(t), f) for n, t, f in FILAS]
for n, t, f in df_tok:
    print(f"{n:55s} {t:>9,d} tokens   [{f}]")
print("\n1 h de audio EnCodec@6kbps =", f"{tokens_audio(3600, 75, 8):,}", "tokens | ventana de 128K tokens =", f"{128_000/tokens_audio(1, 75, 8)/60:.1f} min de audio")

fig, ax = plt.subplots(figsize=(9, 4.2)); nom = [n.split(" (")[0] if len(n) > 40 else n for n, _, _ in df_tok]; val = [t for _, t, _ in df_tok]
cols = [GRIS] + [AZUL] * 5 + [NARANJA] * 3 + [VERDE]
ax.barh(range(len(val)), val, color=cols); ax.set_yticks(range(len(val))); ax.set_yticklabels(nom, fontsize=7); ax.invert_yaxis()
ax.axvline(128_000, color="crimson", ls="--", lw=1, label="ventana de contexto de 128K tokens"); ax.set_xscale("log"); ax.set_xlabel("tokens (escala log)")
for i, v in enumerate(val): ax.text(v * 1.1, i, f"{v:,}", va="center", fontsize=7)
ax.set_title("¿Cuánto contexto cuesta cada modalidad?"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

> **👀 Qué debes observar**
>
> * **El audio es caro:** a 6 kbps EnCodec produce 600 tokens/s; **1 hora = 2.16 M de tokens**, y una ventana de 128K tokens cabe solo ~3.6 minutos. Por eso Mimi baja a **12.5 Hz** (100 tokens/s aplanado) y Moshi modela los niveles con un transformer «de profundidad» pequeño en lugar de aplanarlos.
> * Una imagen discreta de Chameleon (1 024 tokens) cuesta **~5 veces** un parche ViT-B/16 de 224² (196). Y el **vídeo** (decenas de miles de tokens para 5 s) obliga a compresión espacio-temporal agresiva.
> * TiTok (32 tokens por imagen) muestra que 1 024 no es ley natural: depende del tokenizer.

## 6. Tokens continuos: parches ViT

Un **Vision Transformer** parte la imagen en una rejilla de parches $p\times p$, aplana cada uno ($p^2 C$ números) y lo proyecta linealmente a $D$ dimensiones: $N=(H/p)(W/p)$ tokens. Es *exactamente* una convolución con `kernel=p, stride=p`. Es también la «lectura» de una imagen que usan CLIP/SigLIP (módulo 16).

In [ ]:
def patchify(img, p):
    """[B,C,H,W] -> [B, N, C*p*p]: un 'token' continuo por parche p x p (ViT)."""
    B, C, H, W = img.shape
    x = img.unfold(2, p, p).unfold(3, p, p)                        # [B,C,H/p,W/p,p,p]
    return x.permute(0, 2, 3, 1, 4, 5).reshape(B, (H // p) * (W // p), C * p * p)

im, fuente = foto("china.jpg"); im224 = im.resize((224, 224))
x = torch.from_numpy(np.asarray(im224, np.float32) / 255).permute(2, 0, 1)[None]
tok = patchify(x, 16)
D = 192; conv = nn.Conv2d(3, D, 16, stride=16)                      # la 'patch embedding' real de ViT = una convolución
via_conv = conv(x).flatten(2).transpose(1, 2)
via_lineal = tok @ conv.weight.view(D, -1).T + conv.bias
print(f"{fuente} | parches: {tuple(tok.shape)} = (1, {224//16}², 3·16·16) | conv == lineal sobre parches: {torch.allclose(via_conv, via_lineal, atol=1e-5)}")

fig, axs = plt.subplots(1, 3, figsize=(14, 4))
axs[0].imshow(im224); axs[0].set_title("Imagen 224x224 con rejilla de parches 16x16 (196 tokens)", fontsize=9)
for k in range(0, 224, 16): axs[0].axvline(k, color="w", lw=.5); axs[0].axhline(k, color="w", lw=.5)
axs[0].grid(False); axs[0].axis("off")
for j in range(6):
    r, c = divmod(j * 31 + 40, 14); axs[1].imshow(np.asarray(im224)[r*16:(r+1)*16, c*16:(c+1)*16], extent=(j, j + .95, 0, 1));
axs[1].set_xlim(0, 6); axs[1].set_ylim(0, 1); axs[1].set_title("Seis parches = seis 'tokens' (cada uno -> vector de 768 números)", fontsize=9); axs[1].axis("off")
res = np.arange(224, 1281, 32)
for p, col in [(32, VERDE), (16, AZUL), (14, NARANJA), (8, GRIS)]:
    axs[2].plot(res, (res // p) ** 2, color=col, lw=2, label=f"parche {p}")
axs[2].set_yscale("log"); axs[2].set_xlabel("resolución (lado, píxeles)"); axs[2].set_ylabel("tokens por imagen")
axs[2].set_title("Tokens vs resolución (la atención cuesta ~N²)", fontsize=9); axs[2].legend(fontsize=8); plt.tight_layout(); plt.show()

> **👀 Qué debes observar**
>
> * `conv == lineal sobre parches: True`: la *patch embedding* es una **convolución con stride = tamaño de parche**; no hay nada mágico.
> * **Derecha:** el número de tokens crece **cuadráticamente con la resolución** y la atención cuesta ~$N^2$: duplicar el lado cuadruplica los tokens y multiplica por 16 el coste de atención. Por eso los VLMs recortan, remuestrean o agrupan parches.

## 7. Tokens discretos: VQ-VAE

**Idea.** Un autoencoder cuyo cuello de botella es un **codebook** de $K$ vectores $\{e_j\}_{j=1}^{K}\subset\mathbb{R}^d$. El encoder produce $z_e(x)$ (una rejilla de vectores), cada vector se **sustituye por su vecino más cercano** y el decoder reconstruye desde esos vectores:

$$k = \arg\min_{j}\ \lVert z_e - e_j\rVert_2,\qquad z_q = e_k,\qquad \hat x = \text{Dec}(z_q).$$

**Pérdida** (van den Oord et al., 2017), con $\operatorname{sg}$ el *stop-gradient*:

$$\mathcal{L} = \underbrace{\lVert x-\hat x\rVert_2^2}_{\text{reconstrucción}} + \underbrace{\lVert \operatorname{sg}[z_e]-e\rVert_2^2}_{\text{codebook}} + \beta\underbrace{\lVert z_e-\operatorname{sg}[e]\rVert_2^2}_{\text{commitment}},\qquad \beta=0.25$$

El **argmin** no es diferenciable: se usa el **straight-through estimator**, $z_q = z_e + \operatorname{sg}(e - z_e)$, que en *forward* da $e$ y en *backward* copia el gradiente de $z_q$ a $z_e$. La pérdida de codebook mueve los $e_j$ hacia los vectores que se les asignan; el *commitment* evita que el encoder «salte» entre códigos.

**Tasa.** Con una rejilla de $N$ tokens, la tasa es $R = N\log_2 K$ bits por imagen (o $R/(H W)$ bits por píxel, **bpp**).

**Colapso del codebook.** Si solo unos pocos códigos ganan la asignación, el resto **nunca recibe gradiente** (están «muertos»): la tasa efectiva es mucho menor que $\log_2 K$. Se mide con la **perplejidad de uso** $\exp\!\big(-\sum_k p_k\ln p_k\big)$ ($p_k$ = frecuencia del código $k$): vale $K$ si el uso es uniforme y ~1 si colapsa. Soluciones habituales: inicializar el codebook **con vectores del encoder** (como hacemos), **reiniciar códigos muertos**, actualizar por **EMA**, o evitar el codebook (**FSQ**).

> ### 🎨 GEMINI-IMAGEN · ID: `M08-IMG-02` · ⚠️ REEMPLAZAR ESTA CELDA
> **VQ-VAE: encoder, codebook y decoder con straight-through**
>
> **Qué muestra:** Flujo completo de un VQ-VAE: imagen → rejilla de vectores continuos → vecino más cercano en el codebook → rejilla de ids → decoder, con el camino de gradiente straight-through.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Diagrama de VQ-VAE con encoder, búsqueda del vecino más cercano en el codebook, decoder y el camino de gradiente straight-through
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, estilo figura de paper (sans-serif legible, flechas claras, sin ilustraciones decorativas ni texto inventado). Paleta: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Formato 16:9.
> Título: "VQ-VAE: de píxeles a ids y de vuelta".
> De izquierda a derecha, cinco bloques:
> 1) Cuadrado gris etiquetado "Imagen x (28x28)".
> 2) Bloque azul "Encoder (convoluciones, stride 4)" que produce una rejilla de 7x7 cuadraditos azules etiquetada "z_e: 49 vectores de dimensión d".
> 3) En el centro, un recuadro naranja grande "Codebook: K vectores e₁ … e_K" mostrado como una columna de K filas (con "e₁", "e₂", "…", "e_K"). Desde un cuadradito de la rejilla azul sale una flecha etiquetada "vecino más cercano: k = argmin ‖z_e − e_j‖" hasta la fila "e₅" resaltada, y de ahí a un cuadradito naranja de una segunda rejilla 7x7 etiquetada "z_q = e_k". Debajo de la segunda rejilla, una tercera rejilla 7x7 de números (por ejemplo 5, 17, 17, 3, …) etiquetada "ids: 49 tokens (log₂K bits cada uno)".
> 4) Bloque verde "Decoder (deconvoluciones)" que produce un cuadrado gris etiquetado "Reconstrucción x̂".
> 5) Debajo de todo, una flecha curva discontinua roja que va desde z_q de vuelta hasta z_e con la etiqueta "straight-through: el gradiente se copia de z_q a z_e (el argmin no es diferenciable)".
> Pie con las tres pérdidas en tres cajitas: "‖x − x̂‖²" (reconstrucción), "‖sg[z_e] − e‖²" (codebook), "β‖z_e − sg[e]‖²" (commitment).
> Evita: ilustraciones decorativas, texto ilegible, números no indicados, flechas ambiguas, sombras 3D, logotipos.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M08-IMG-02.png` en la carpeta del módulo y reemplaza esta celda por: `![Diagrama de VQ-VAE con encoder, búsqueda del vecino más cercano en el codebook, decoder y el camino de gradiente straight-through](assets/M08-IMG-02.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> x (28x28) -> [Encoder] -> z_e (7x7 x d) --argmin--> ids (7x7, K valores) -> e[ids] = z_q -> [Decoder] -> x̂
>                                    \__________ straight-through: grad(z_q) -> z_e __________/
> pérdidas: ||x-x̂||² + ||sg[z_e]-e||² + β||z_e - sg[e]||²
> ```

> **👀 Qué debes observar**
>
> Hay **tres** caminos: (1) el de *reconstrucción* (a través de $z_q$ con STE hasta el encoder), (2) el del **codebook** (mueve los $e_j$ hacia los $z_e$) y (3) el del **commitment** (mantiene a $z_e$ cerca del código elegido). Sin el STE el encoder no recibiría gradiente de la reconstrucción.

### 7.1 Cuantizadores desde cero

Los tres cuantizadores (`VQ`, `FSQ`, `RVQ`) comparten interfaz.

In [ ]:
class VQ(nn.Module):
    """Cuantizador vectorial (van den Oord et al., 2017). z: [B,d,*] -> zq (con STE), pérdida, índices [B,*]."""
    def __init__(self, K, d, beta=0.25):
        super().__init__(); self.K, self.d, self.beta = K, d, beta
        self.cb = nn.Embedding(K, d); self.cb.weight.data.uniform_(-1 / K, 1 / K)
        self.register_buffer("inicializado", torch.tensor(False)); self.register_buffer("uso", torch.ones(K))
    def forward(self, z, reinicia=False):
        shape = z.shape; zf = z.movedim(1, -1).reshape(-1, self.d)
        if self.training and not bool(self.inicializado):                       # init por datos: codebook = vectores del encoder
            self.cb.weight.data.copy_(zf[torch.randperm(len(zf), device=z.device)[:self.K]].detach()); self.inicializado.fill_(True)
        dist = zf.pow(2).sum(1, keepdim=True) - 2 * zf @ self.cb.weight.T + self.cb.weight.pow(2).sum(1)
        idx = dist.argmin(1); e = self.cb(idx)
        loss = F.mse_loss(e, zf.detach()) + self.beta * F.mse_loss(zf, e.detach())     # ||sg[z]-e||² + β||z-sg[e]||²
        if self.training:
            self.uso.mul_(0.99).add_(torch.bincount(idx, minlength=self.K).float(), alpha=0.01 * 1.0)
            if reinicia:                                                                  # reinicio de códigos muertos
                muertos = self.uso < 1e-3 * len(idx) / self.K
                if muertos.any(): self.cb.weight.data[muertos] = zf[torch.randint(len(zf), (int(muertos.sum()),), device=z.device)].detach()
        zq = zf + (e - zf).detach()                                                       # straight-through estimator
        return zq.reshape(*shape[:1], *shape[2:], self.d).movedim(-1, 1), loss, idx.reshape(shape[0], *shape[2:])

class FSQ(nn.Module):
    """Finite Scalar Quantization (Mentzer et al., 2023): acota cada canal y lo redondea a L_i niveles; sin codebook ni pérdidas extra.
    (Normalizamos z por su desviación típica por canal: con salidas del encoder ~0.1 todo redondearía a 0 y el entrenamiento colapsa.)"""
    def __init__(self, levels):
        super().__init__(); L = torch.tensor(levels); self.register_buffer("L", L); self.d = len(levels)
        self.register_buffer("basis", torch.cumprod(torch.tensor([1] + list(levels[:-1])), 0)); self.K = int(L.prod())
    def forward(self, z, reinicia=False):
        zp = z.movedim(1, -1); zp = zp / (zp.detach().reshape(-1, self.d).std(0) + 1e-5); half = (self.L - 1) * (1 - 1e-3) / 2; off = torch.where(self.L % 2 == 0, 0.5, 0.0); shift = torch.atanh(off / half)
        b = torch.tanh(zp + shift) * half - off; q = b + (torch.round(b) - b).detach()          # acota + redondea (STE)
        hw = (self.L // 2).float(); idx = ((q + hw) * self.basis).sum(-1).long()
        return (q / hw).movedim(-1, 1), z.new_zeros(()), idx

class RVQ(nn.Module):
    """Cuantización vectorial residual: cada nivel cuantiza lo que el anterior dejó sin explicar."""
    def __init__(self, Q, K, d, beta=0.25):
        super().__init__(); self.Q, self.K = Q, K; self.capas = nn.ModuleList([VQ(K, d, beta) for _ in range(Q)])
    def forward(self, z, reinicia=False, n_q=None):
        res, out, loss, idxs = z, 0, 0, []
        for capa in self.capas[: (n_q or self.Q)]:
            q, l, idx = capa(res, reinicia); res = res - q.detach(); out = out + q; loss = loss + l; idxs.append(idx)
        return out, loss, torch.stack(idxs, 1)

def perplexity(idx, K):
    p = torch.bincount(idx.flatten(), minlength=K).float(); p = p / p.sum()
    return float(torch.exp(-(p[p > 0] * p[p > 0].log()).sum()))

def psnr(x, y, peak=1.0):
    mse = F.mse_loss(x, y).item(); return 10 * math.log10(peak ** 2 / max(mse, 1e-10))

In [ ]:
class AE2D(nn.Module):
    """Autoencoder convolucional 28x28 -> 7x7 con un cuantizador enchufable (VQ / FSQ / RVQ)."""
    def __init__(self, quant, d, ch=32):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(1, ch, 4, 2, 1), nn.ReLU(), nn.Conv2d(ch, 2 * ch, 4, 2, 1), nn.ReLU(), nn.Conv2d(2 * ch, d, 1))
        self.quant = quant
        self.dec = nn.Sequential(nn.Conv2d(d, 2 * ch, 3, 1, 1), nn.ReLU(), nn.ConvTranspose2d(2 * ch, ch, 4, 2, 1), nn.ReLU(),
                                 nn.ConvTranspose2d(ch, 1, 4, 2, 1), nn.Sigmoid())
    def forward(self, x, reinicia=False, **kw):
        z = self.enc(x); zq, ql, idx = self.quant(z, reinicia, **kw); return self.dec(zq), ql, idx

def entrena_ae(modelo, pasos, B=128, lr=2e-3, seed=SEED, reinicia_cada=50, **kw):
    g = torch.Generator().manual_seed(seed); opt = torch.optim.Adam(modelo.parameters(), lr=lr); hist = []
    for s in range(pasos):
        modelo.train(); ix = torch.randint(0, len(X_TR), (B,), generator=g).to(device); x = X_TR[ix]
        xr, ql, idx = modelo(x, reinicia=(reinicia_cada and s % reinicia_cada == 0 and s > 0), **kw)
        rec = F.mse_loss(xr, x); loss = rec + ql; opt.zero_grad(); loss.backward(); opt.step()
        if s % 20 == 0: hist.append((s, rec.item(), perplexity(idx[:, 0] if idx.dim() == 4 else idx, modelo.quant.K)))
    return hist

@torch.no_grad()
def evalua_ae(modelo, **kw):
    modelo.eval(); xr, _, idx = modelo(X_TE, **kw)
    return {"psnr": psnr(xr, X_TE), "ppl": perplexity(idx[:, 0] if idx.dim() == 4 else idx, modelo.quant.K), "idx": idx, "xr": xr}

### 7.2 Entrenar un VQ-VAE ($K=64$, 49 tokens por imagen)

In [ ]:
PASOS_AE = 400 if FAST_DEV_RUN else 3000
torch.manual_seed(SEED); t0 = time.time()
K0, D0 = 64, 16
vq = AE2D(VQ(K0, D0), D0).to(device); h_vq = entrena_ae(vq, PASOS_AE); r_vq = evalua_ae(vq)
tokens_por_img = r_vq["idx"].shape[1] * r_vq["idx"].shape[2]
print(f"VQ-VAE K={K0}: {time.time()-t0:.0f}s | PSNR test = {r_vq['psnr']:.2f} dB | perplejidad = {r_vq['ppl']:.1f} de {K0} | "
      f"{tokens_por_img} tokens/img = {tokens_por_img*math.log2(K0):.0f} bits (vs {28*28*8} bits en bruto: {28*28*8/(tokens_por_img*math.log2(K0)):.0f}x menos)")

fig, axs = plt.subplots(3, 8, figsize=(13, 5.6))
for j in range(8):
    axs[0, j].imshow(X_TE[j, 0].cpu(), cmap="gray"); axs[1, j].imshow(r_vq["xr"][j, 0].cpu(), cmap="gray")
    axs[2, j].imshow(r_vq["idx"][j].cpu(), cmap="tab20", vmin=0, vmax=K0)
    for i in range(7):
        for k in range(7): axs[2, j].text(k, i, int(r_vq["idx"][j, i, k]), ha="center", va="center", fontsize=4.5)
    for i in range(3): axs[i, j].axis("off")
for i, t in enumerate(["original", "reconstrucción desde 49 tokens", "mapa de tokens 7x7 (ids)"]): axs[i, 0].set_title(t, fontsize=8, loc="left")
plt.tight_layout(); plt.show()

> **👀 Qué debes observar**
>
> * **Fila central:** la imagen se reconstruye desde solo **49 ids** (¡de ~6 300 bits en bruto a ~300!), con detalles borrosos: es una compresión **con pérdida** muy agresiva.
> * **Fila inferior:** el mapa de tokens 7x7 tiene **estructura espacial**: trazos parecidos reutilizan los mismos ids. Eso es lo que un LM puede aprender a predecir después.
> * **Perplejidad impresa:** si es muy inferior a $K$, el codebook no se usa por completo (siguiente celda).

### 7.3 Barrido de $K$ y colapso del codebook

Entrenamos varios $K$ y un caso **sin los trucos** de uso (sin init por datos ni reinicio de códigos muertos) para ver el colapso.

In [ ]:
# Barrido de K (tamaño del codebook) + el fenómeno de colapso: sin trucos vs con init por datos + reinicio de códigos muertos
class VQsimple(VQ):
    """VQ SIN los trucos de uso (sin init por datos, sin reinicio): codebook inicial pequeño y aleatorio."""
    def forward(self, z, reinicia=False):
        self.inicializado.fill_(True); return super().forward(z, reinicia=False)

res_K, t0 = {}, time.time()
for K in [4, 16, 64, 256, 1024]:
    torch.manual_seed(SEED); m = AE2D(VQ(K, D0), D0).to(device); entrena_ae(m, PASOS_AE); r = evalua_ae(m); res_K[K] = (r["psnr"], r["ppl"], 49 * math.log2(K))
torch.manual_seed(SEED); m_c = AE2D(VQsimple(256, D0), D0).to(device); entrena_ae(m_c, PASOS_AE, reinicia_cada=0); r_c = evalua_ae(m_c)
print(f"({time.time()-t0:.0f}s)")
for K, (p, pp, b) in res_K.items(): print(f"K={K:5d}: PSNR={p:5.2f} dB | perplejidad={pp:6.1f} ({100*pp/K:5.1f}% del codebook 'efectivo') | {b:6.0f} bits/img")
print(f"K=256 SIN trucos: PSNR={r_c['psnr']:.2f} dB | perplejidad={r_c['ppl']:.1f}")

fig, axs = plt.subplots(1, 3, figsize=(14, 3.8))
Ks = list(res_K); axs[0].plot([res_K[k][2] for k in Ks], [res_K[k][0] for k in Ks], "o-", color=AZUL, lw=2)
for k in Ks: axs[0].annotate(f"K={k}", (res_K[k][2], res_K[k][0]), fontsize=7, xytext=(3, -10), textcoords="offset points")
axs[0].set_xlabel("bits por imagen (49 tokens x log2 K)"); axs[0].set_ylabel("PSNR test (dB)"); axs[0].set_title("Tasa-distorsión del VQ-VAE")
axs[1].bar([str(k) for k in Ks], [res_K[k][1] / k for k in Ks], color=AZUL); axs[1].set_ylabel("perplejidad / K"); axs[1].set_xlabel("K"); axs[1].set_title("Uso efectivo del codebook (1 = uniforme)")
u1 = torch.bincount(r_vq["idx"].flatten(), minlength=K0).cpu().numpy(); u2 = torch.bincount(r_c["idx"].flatten(), minlength=256).cpu().numpy()
axs[2].bar(range(256), np.sort(u2)[::-1], color=NARANJA, label="K=256 sin trucos"); axs[2].set_yscale("log"); axs[2].set_xlabel("códigos ordenados por frecuencia"); axs[2].set_ylabel("usos en test"); axs[2].set_title("Colapso del codebook"); axs[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

> **👀 Qué debes observar**
>
> * **Izquierda (tasa-distorsión):** más bits (más $K$) → más PSNR, con **rendimientos decrecientes**. El codo marca el compromiso razonable.
> * **Centro:** perplejidad/$K$ **cae** al crecer $K$: con $K$ grande una parte del codebook se desperdicia aunque hayamos usado trucos. No pagas $\log_2 K$ bits «útiles» por token.
> * **Derecha:** sin trucos, **unos pocos códigos concentran todo el uso** (curva en escala log): el resto está muerto. Es el fallo más común al entrenar un VQ-VAE por primera vez.

## 8. FSQ y RVQ: dos maneras de esquivar el codebook (o de apilarlo)

**FSQ — *Finite Scalar Quantization*** (Mentzer et al., 2023). Elimina el codebook: acota cada canal de $z$ con una $\tanh$ y lo **redondea a $L_i$ niveles** enteros (con STE para el redondeo):

$$\hat z_i = \operatorname{round}\big(\lfloor L_i/2\rfloor\tanh(z_i)\big),\qquad |\mathcal{C}| = \prod_{i=1}^{d}L_i\quad(\text{p. ej. } [8,6,5]\Rightarrow 240\text{ códigos}).$$

El «codebook» es **implícito** (una rejilla de enteros), no hay pérdidas auxiliares y el uso es casi uniforme. Los tokenizers discretos de Cosmos usan niveles $(8,8,8,5,5,5)$, es decir **64 000** códigos.

**RVQ — *Residual Vector Quantization*** (SoundStream/EnCodec). Apila $Q$ cuantizadores, cada uno sobre el **residuo** del anterior:

$$r_0 = z,\quad q_i = \text{VQ}_i(r_{i-1}),\quad r_i = r_{i-1}-q_i,\qquad \hat z=\sum_{i=1}^{Q}q_i,\qquad \text{tasa}=Q\log_2 K\ \text{bits por posición}.$$

Los primeros niveles capturan lo grueso; los siguientes **refinan**. Entrenando con ***quantizer dropout*** (usar un $Q$ aleatorio por lote, como en SoundStream) **un solo modelo sirve varias tasas**: recortas niveles en inferencia.

In [ ]:
t0 = time.time(); R = {}
torch.manual_seed(SEED); m = AE2D(VQ(256, D0), D0).to(device); entrena_ae(m, PASOS_AE); r = evalua_ae(m); R["VQ K=256"] = (r["psnr"], r["ppl"], 256, 49 * 8)
torch.manual_seed(SEED); fsq = FSQ([8, 6, 5]); m = AE2D(fsq, 3).to(device); entrena_ae(m, PASOS_AE, lr=1e-3);   # FSQ: lr menor (con 2e-3 colapsa en esta semilla)
r = evalua_ae(m); R["FSQ [8,6,5]"] = (r["psnr"], r["ppl"], fsq.K, 49 * math.log2(fsq.K))
torch.manual_seed(SEED); rvq = RVQ(4, 16, D0); m_rvq = AE2D(rvq, D0).to(device)
def paso_rvq(m, pasos, B=128, lr=2e-3):                         # entrenamiento con 'quantizer dropout': n_q aleatorio por lote
    opt = torch.optim.Adam(m.parameters(), lr=lr); g = torch.Generator().manual_seed(SEED)
    for s in range(pasos):
        m.train(); ix = torch.randint(0, len(X_TR), (B,), generator=g).to(device); x = X_TR[ix]; nq = int(torch.randint(1, 5, (1,), generator=g))
        xr, ql, _ = m(x, reinicia=(s % 50 == 0 and s > 0), n_q=nq); loss = F.mse_loss(xr, x) + ql; opt.zero_grad(); loss.backward(); opt.step()
paso_rvq(m_rvq, PASOS_AE)
RV = {nq: evalua_ae(m_rvq, n_q=nq) for nq in [1, 2, 3, 4]}
for nq, r in RV.items(): print(f"RVQ con {nq} niveles (K=16): PSNR={r['psnr']:.2f} dB | {49*4*nq} bits/img")
print({k: f"PSNR={v[0]:.2f}, perplejidad={v[1]:.0f}/{v[2]}" for k, v in R.items()}, f"({time.time()-t0:.0f}s)")

fig, axs = plt.subplots(1, 3, figsize=(14, 3.8)); b0 = [R[k][3] for k in R]
axs[0].bar(list(R), [R[k][0] for k in R], color=[AZUL, VERDE]); axs[0].set_ylim(min(R[k][0] for k in R) - 2, None); axs[0].set_ylabel("PSNR test (dB)"); axs[0].set_title("VQ vs FSQ (~256 códigos)")
axs[1].bar(list(R), [100 * R[k][1] / R[k][2] for k in R], color=[AZUL, VERDE]); axs[1].set_ylabel("perplejidad / |C| (%)"); axs[1].set_title("Uso del 'vocabulario' de códigos")
axs[2].plot([49 * 4 * nq for nq in RV], [RV[nq]["psnr"] for nq in RV], "o-", color=NARANJA, lw=2, label="RVQ (Q niveles, K=16)")
axs[2].plot([res_K[k][2] for k in [4, 16, 64, 256]], [res_K[k][0] for k in [4, 16, 64, 256]], "s--", color=AZUL, label="VQ simple (barrido de K)")
axs[2].set_xlabel("bits por imagen"); axs[2].set_ylabel("PSNR (dB)"); axs[2].set_title("Un único modelo RVQ cubre varias tasas"); axs[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

> **👀 Qué debes observar**
>
> * **VQ vs FSQ (izquierda y centro):** con un presupuesto parecido de códigos (~256 vs 240) FSQ alcanza una calidad comparable **sin codebook, sin pérdidas auxiliares y sin trucos** (init por datos, reinicios). En la corrida de referencia: VQ 22.3 dB con perplejidad 22/256 y FSQ 21.2 dB con 41/240 — FSQ usa **casi el doble** de su vocabulario, aunque ninguno lo llena (MNIST 7x7 no tiene variedad para cientos de códigos); el paper de FSQ reporta uso casi completo a escala. Dos detalles prácticos que nos costó descubrir: **normalizamos** la entrada de FSQ por su desviación típica (si no, todo redondea a 0) y usamos un **learning rate menor** (con 2e-3 el autoencoder colapsaba a una salida constante en esta semilla). FSQ no tiene codebook, pero sigue siendo sensible a la escala.
> * **Derecha:** un único modelo **RVQ** con `n_q = 1…4` traza una **curva de tasas** (más niveles → más PSNR). El VQ simple necesita **un modelo por punto** de la curva.
> * ⚠️ Una semilla, ~400 pasos, MNIST: las diferencias finas (±0.5 dB) **no son concluyentes**; busca las tendencias.

## 9. Tokens de imagen dentro de un LLM (estilo Chameleon) y modelos «continuos»

**Chameleon** (Meta, 2024) entrena un transformer *early-fusion* sobre **secuencias mixtas** texto+imagen en un único vocabulario: un tokenizer de imagen convierte una imagen de 512x512 en **1 024 tokens** de un codebook de **8 192**, y el vocabulario total (BPE de texto + tokens de imagen) es de **65 536** entradas. **Emu3** (BAAI, 2024) lleva la idea a «solo predecir el siguiente token» para texto, imagen y vídeo.

**¿Por qué discretizar?** Porque permite usar la **misma pérdida** (cross-entropy del siguiente token) y la misma arquitectura para todo, y mezclar modalidades en una secuencia. **¿Por qué no?** La cuantización **pierde información** y limita la calidad; Transfusion (Meta, hasta 7B) combina cross-entropy para texto con **difusión** sobre parches *continuos* y reporta que escala **mejor** que cuantizar las imágenes; MAR (Li et al. 2024) modela cada token continuo con una **pérdida de difusión** y «elimina la necesidad de tokenizers discretos».

Aquí construimos, a escala de juguete, la receta discreta: **un LM sobre los 49 tokens de cada imagen**, con un token de *clase* como «prompt». Es el mismo principio de VQGAN + transformer (Esser et al., 2020), sin GAN ni perceptual loss.

In [ ]:
# Mini-Chameleon: un LM sobre tokens de imagen. Secuencia = [token de clase] + 49 tokens VQ; vocabulario unificado (códigos + clases).
class TinyGPT(nn.Module):
    def __init__(self, vocab, d=96, L=3, ctx=64, H=4):
        super().__init__(); self.emb, self.pos = nn.Embedding(vocab, d), nn.Embedding(ctx, d)
        layer = nn.TransformerEncoderLayer(d, H, 4 * d, dropout=0.0, batch_first=True, norm_first=True, activation="gelu")
        self.tr = nn.TransformerEncoder(layer, L, enable_nested_tensor=False); self.ln, self.head = nn.LayerNorm(d), nn.Linear(d, vocab, bias=False)
    def forward(self, x):
        T = x.shape[1]; h = self.emb(x) + self.pos(torch.arange(T, device=x.device)); m = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), 1)
        return self.head(self.ln(self.tr(h, mask=m, is_causal=True)))

with torch.no_grad():
    vq.eval(); idx_tr = torch.cat([vq(X_TR[i:i + 500])[2].flatten(1) for i in range(0, len(X_TR), 500)])        # [N,49] tokens de cada imagen
SEQ = torch.cat([(K0 + Y_TR).unsqueeze(1), idx_tr], 1)                                                         # prefijo = "palabra" de clase
V = K0 + N_CLASES
gpt = TinyGPT(V, ctx=SEQ.shape[1]).to(device); opt = torch.optim.AdamW(gpt.parameters(), lr=2e-3); g = torch.Generator().manual_seed(SEED)
t0 = time.time(); PASOS_P = 400 if FAST_DEV_RUN else 3000
for s in range(PASOS_P):
    b = SEQ[torch.randint(0, len(SEQ), (64,), generator=g).to(device)]; loss = F.cross_entropy(gpt(b[:, :-1]).reshape(-1, V), b[:, 1:].reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
print(f"LM sobre tokens de imagen: {time.time()-t0:.0f}s | NLL final = {loss.item():.2f} nats/token (ruido uniforme: {math.log(K0):.2f})")

@torch.no_grad()
def genera(clase, n=8, temp=0.9):
    x = torch.full((n, 1), K0 + clase, device=device)
    for _ in range(49):
        lg = gpt(x)[:, -1, :K0] / temp; x = torch.cat([x, torch.multinomial(F.softmax(lg, -1), 1)], 1)
    z = vq.quant.cb(x[:, 1:]).view(n, 7, 7, D0).permute(0, 3, 1, 2); return vq.dec(z).cpu()

fig, axs = plt.subplots(N_CLASES if N_CLASES <= 4 else 4, 8, figsize=(10, 5.2 if N_CLASES > 3 else 4)); filas = axs if axs.ndim > 1 else axs[None]
for c in range(filas.shape[0]):
    imgs = genera(c)
    for j in range(8): filas[c, j].imshow(imgs[j, 0], cmap="gray"); filas[c, j].axis("off")
    filas[c, 0].set_title(f"clase {c} (token de 'prompt')", fontsize=8, loc="left")
plt.tight_layout(); plt.show()

> **👀 Qué debes observar**
>
> Cada fila se genera con un **token de clase** como prefijo y 49 tokens muestreados autoregresivamente, que el decoder del VQ-VAE traduce a píxeles. Verás trazos plausibles pero **borrosos o con artefactos**: tiene dos fuentes de error (el LM sobre tokens y la capacidad del VQ-VAE), exactamente las que un VQGAN real mitiga con más escala, pérdidas perceptual/adversarial y codebooks mayores. La idea que importa: **una vez discretizada la imagen, generarla es «solo» un LM** sobre un vocabulario unificado.

## 10. Audio: de la forma de onda a tokens con un códec neuronal

**Pipeline** (SoundStream, EnCodec, DAC, Mimi): `waveform → encoder convolucional (stride total S) → RVQ (Q niveles) → decoder → waveform`. A una frecuencia de muestreo $f_s$, el encoder entrega $f_s/S$ *frames* por segundo, y

$$\text{tasa} = \frac{f_s}{S}\cdot Q\cdot\log_2 K\ \ \text{bit/s},\qquad \text{tokens/s}= \frac{f_s}{S}\cdot Q .$$

| Códec | Frecuencia | Frames/s | Cuantización | Tasa |
|---|---|---|---|---|
| **SoundStream** (Google, 2021) | 24 kHz | 75 (strides 2,4,5,8) | RVQ con *quantizer dropout* | 3-18 kbps |
| **EnCodec** (Meta, 2022) | 24 kHz | 75 | RVQ hasta 32 codebooks de 1 024 entradas (10 bits) | 1.5, 3, 6, 12, 24 kbps |
| **DAC** (Descript, 2023) | 44.1 kHz | — | RVQGAN mejorado | 8 kbps (~90x de compresión) |
| **Mimi** (Kyutai, 2024) | 24 kHz | **12.5** | 8 cuantizadores de 2 048 entradas (11 bits); el primero **destila** información semántica (WavLM) | **1.1 kbps** |

* **Verificación aritmética:** EnCodec 24 kHz con 8 codebooks: $75\times 8\times 10 = 6\,000$ bit/s ✓. Mimi: $12.5\times 8\times 11 = 1\,100$ bit/s ✓.
* **Tokens acústicos vs semánticos.** Los niveles de un códec puro capturan *cómo suena* (timbre, ruido). Mimi y SpeechTokenizer añaden **destilación de un modelo de habla** (WavLM) en el primer nivel para que contenga *qué se dice*; así un LM puede predecir el contenido con los primeros tokens y los detalles acústicos con los siguientes.
* **Entrenamiento:** reconstrucción en forma de onda + **pérdida espectral multi-escala** + **discriminadores** (GAN) + pérdidas de commitment del RVQ. Nosotros usamos solo las dos primeras (no hay GAN), de ahí un audio con artefactos.

> ### 🎨 GEMINI-IMAGEN · ID: `M08-IMG-03` · ⚠️ REEMPLAZAR ESTA CELDA
> **Códec de audio neuronal con RVQ (EnCodec/SoundStream)**
>
> **Qué muestra:** Arquitectura de un códec de audio neuronal: encoder convolucional, RVQ de varios niveles y decoder; con la cuenta de tokens por segundo.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Diagrama de códec neuronal: audio, encoder convolucional, cascada de 8 cuantizadores RVQ, matriz de tokens y decoder
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, estilo figura de paper (sans-serif legible, flechas claras, sin ilustraciones decorativas ni texto inventado). Paleta: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Formato 16:9.
> Título: "Códec neuronal de audio: waveform → tokens → waveform".
> De izquierda a derecha:
> 1) Una onda sonora gris etiquetada "Audio 24 kHz (1.44 M muestras por minuto)".
> 2) Bloque azul "Encoder convolucional (strides 2·4·5·8 = 320)" con una flecha etiquetada "75 frames por segundo".
> 3) Un bloque naranja grande "RVQ: 8 cuantizadores en cascada". Dentro, 8 cajitas apiladas "VQ 1 (1 024 entradas)", "VQ 2", "…", "VQ 8", unidas por flechas verticales etiquetadas "residuo". A la derecha del bloque, una matriz de 8 filas x 5 columnas de cajitas pequeñas con ids (filas = niveles, columnas = frames), etiquetada "tokens: 75 x 8 = 600 por segundo".
> 4) Bloque verde "Decoder convolucional" que reconstruye una onda gris etiquetada "Audio reconstruido".
> Debajo, una línea horizontal con tres marcas: "1 nivel → baja calidad", "4 niveles → media", "8 niveles → 6 kbps" (con la cuenta "75 × 8 × 10 bits = 6 000 bit/s") y la leyenda "quantizer dropout: un solo modelo, varias tasas".
> Nota lateral en gris: "Mimi: 12.5 frames/s x 8 niveles de 2 048 entradas = 1.1 kbps".
> Evita: ilustraciones decorativas, texto ilegible, números distintos de los indicados, flechas ambiguas, sombras 3D, logotipos.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M08-IMG-03.png` en la carpeta del módulo y reemplaza esta celda por: `![Diagrama de códec neuronal: audio, encoder convolucional, cascada de 8 cuantizadores RVQ, matriz de tokens y decoder](assets/M08-IMG-03.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> audio 24 kHz -> [Encoder strides 2*4*5*8=320] -> 75 frames/s -> [RVQ: VQ1 -> VQ2 -> ... -> VQ8 (residuos)] -> ids 8 x frames
> ids -> [Decoder] -> audio reconstruido     |  75 x 8 x 10 bits = 6000 bit/s  |  quantizer dropout: 1..8 niveles
> ```

> **👀 Qué debes observar**
>
> El orden de los niveles **importa**: el 1.º captura lo más energético (envolvente, tono) y los demás añaden detalle; por eso se pueden **truncar niveles** para bajar la tasa. La matriz de tokens (niveles x frames) es lo que luego modelan los LMs de audio (MusicGen, Moshi…), ya sea **aplanándola** (más tokens) o con un transformer extra para los niveles.

### 10.1 Voz real a 8 kHz y un códec de juguete

Usamos el **Free Spoken Digit Dataset** (FSDD: dígitos hablados en inglés por 6 hablantes, 8 kHz mono, CC BY-SA 4.0), que se descarga de su repositorio oficial en GitHub en segundos; concatenamos las grabaciones y cortamos clips de 1 s. Si no hay red, el respaldo es **voz sintética** (pulsos glotales + formantes de vocales + envolvente silábica) con aviso visible. Es un banco **didáctico** y pequeño: LibriSpeech tiene mucha más variedad.

In [ ]:
from scipy.signal import lfilter
SR = 8000; DUR = 1.0; N_AUD = int(SR * DUR)
VOCALES = {"a": (730, 1090, 2440), "e": (530, 1840, 2480), "i": (270, 2290, 3010), "o": (570, 840, 2410), "u": (300, 870, 2240)}

def resonador(x, f, bw=80):
    """Filtro de 2º orden (formante) a la frecuencia f con ancho de banda bw."""
    r = np.exp(-np.pi * bw / SR); th = 2 * np.pi * f / SR
    return lfilter([1 - r], [1, -2 * r * np.cos(th), r * r], x)

def voz_sintetica(rng, dur=DUR):
    """Clip 'tipo voz': tren de pulsos glotales con f0 variable + formantes de vocales + envolvente silábica + ruido."""
    n = int(SR * dur); t = np.arange(n) / SR
    f0 = rng.uniform(100, 220) * (1 + 0.05 * np.sin(2 * np.pi * rng.uniform(3, 6) * t) + 0.15 * (t / dur - .5))
    fase = np.cumsum(f0) / SR; src = np.sign(np.diff(np.floor(fase), prepend=0)) * 1.0           # pulsos
    src = lfilter([1], [1, -0.97], src)                                                          # espectro -6 dB/oct
    y = np.zeros(n); seg = int(SR * rng.uniform(0.18, 0.3)); i = 0
    while i < n:
        F1, F2, F3 = VOCALES[rng.choice(list(VOCALES))]; j = min(n, i + seg)
        s = src[i:j]; v = resonador(s, F1) + 0.6 * resonador(s, F2, 100) + 0.3 * resonador(s, F3, 120)
        env = np.hanning(len(v)) ** 0.5; y[i:j] += v * env; i = j
    y += rng.randn(n) * 0.01 * np.abs(y).max(); y = y / (np.abs(y).max() + 1e-8) * 0.8
    return y.astype(np.float32)

def carga_fsdd(n_tr, n_te, seed=SEED):
    """Voz REAL: Free Spoken Digit Dataset (6 hablantes, dígitos en inglés, 8 kHz mono, CC BY-SA 4.0), desde su repositorio
    oficial en GitHub. Las grabaciones (~0.4 s) se concatenan y se cortan en clips de 1 s; test = otras grabaciones (índices 45-49)."""
    import io, urllib.request, concurrent.futures as cf
    from scipy.io import wavfile
    base = "https://raw.githubusercontent.com/Jakobovski/free-spoken-digit-dataset/master/recordings/"
    spk = ["george", "jackson", "lucas", "nicolas", "theo", "yweweler"]
    lista = lambda idx: [f"{d}_{s}_{i}.wav" for d in range(10) for s in spk for i in idx]
    def baja(nombre):
        try:
            with urllib.request.urlopen(base + nombre, timeout=20) as r:
                sr, x = wavfile.read(io.BytesIO(r.read()))
        except Exception:
            return None
        x = x.astype(np.float32)
        return x / (np.abs(x).max() + 1e-8) * 0.8 if sr == SR and x.ndim == 1 else None
    def clips(nombres, n):
        with cf.ThreadPoolExecutor(16) as ex:
            xs = [x for x in ex.map(baja, nombres) if x is not None]
        sig = np.concatenate(xs); k = min(n, len(sig) // N_AUD)
        return sig[: k * N_AUD].reshape(k, N_AUD)
    r = np.random.RandomState(seed + 1)                                     # rng propio: no altera el respaldo sintético
    tr, te = list(r.permutation(lista(range(45)))), list(r.permutation(lista(range(45, 50))))
    return clips(tr[: int(2.6 * n_tr) + 10], n_tr), clips(te[: int(2.6 * n_te) + 10], n_te)

rng = np.random.RandomState(SEED)
N_TR_A = 192 if FAST_DEV_RUN else 2000                                       # sin FAST, FSDD da ~1 200 s de train (se usan todos)
try:
    a_tr, a_te = carga_fsdd(N_TR_A, 24)
    assert len(a_tr) >= min(0.9 * N_TR_A, 500) and len(a_te) >= 12, "descarga incompleta"
    A_TR = torch.from_numpy(a_tr)[:, None].to(device); A_TE = torch.from_numpy(a_te)[:, None].to(device)
    FUENTE_AUD = "voz REAL (Free Spoken Digit Dataset, GitHub)"
except Exception as e:
    print("⚠️  [aviso] FSDD no disponible (", type(e).__name__, ") -> voz SINTÉTICA 'tipo voz' (mismo formato; cifras NO comparables con voz real)")
    A_TR = torch.from_numpy(np.stack([voz_sintetica(rng) for _ in range(N_TR_A)]))[:, None].to(device)
    A_TE = torch.from_numpy(np.stack([voz_sintetica(rng) for _ in range(24)]))[:, None].to(device)
    FUENTE_AUD = "audio sintético 'tipo voz'"
print(f"{FUENTE_AUD}: train={tuple(A_TR.shape)} test={tuple(A_TE.shape)} a {SR} Hz  (PCM 16 bit = {SR*16/1000:.0f} kbps)")

In [ ]:
class CodecToy(nn.Module):
    """Códec neuronal de juguete (SoundStream/EnCodec en miniatura): encoder conv (stride total 64) -> RVQ -> decoder conv."""
    def __init__(self, d=32, Q=4, K=256):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv1d(1, 16, 7, 1, 3), nn.ELU(), nn.Conv1d(16, 32, 4, 2, 1), nn.ELU(), nn.Conv1d(32, 64, 8, 4, 2), nn.ELU(),
                                 nn.Conv1d(64, d, 16, 8, 4))
        self.rvq = RVQ(Q, K, d)
        self.dec = nn.Sequential(nn.ConvTranspose1d(d, 64, 16, 8, 4), nn.ELU(), nn.ConvTranspose1d(64, 32, 8, 4, 2), nn.ELU(),
                                 nn.ConvTranspose1d(32, 16, 4, 2, 1), nn.ELU(), nn.Conv1d(16, 1, 7, 1, 3))
    def forward(self, x, reinicia=False, n_q=None):
        zq, ql, idx = self.rvq(self.enc(x), reinicia, n_q); return self.dec(zq), ql, idx

def stft_mag(x, n_fft):
    return torch.stft(x.squeeze(1), n_fft, n_fft // 4, window=torch.hann_window(n_fft, device=x.device), return_complex=True).abs()

def perdida_audio(x, xr):
    """L1 en la forma de onda + L1 de log-magnitud STFT multi-escala (la idea de EnCodec/DAC; sin discriminador)."""
    l = 10 * F.l1_loss(xr, x)                                   # peso 10: sin él el códec ignora la fase (SNR < 0 dB)
    for n in (512, 256, 128): l = l + 0.5 * F.l1_loss(torch.log(stft_mag(xr, n) + 1e-5), torch.log(stft_mag(x, n) + 1e-5))
    return l

def snr_db(x, xr): return float(10 * torch.log10(x.pow(2).sum() / (x - xr).pow(2).sum().clamp_min(1e-10)))
def lsd_db(x, xr, n=512):
    a, b = 20 * torch.log10(stft_mag(x, n) + 1e-5), 20 * torch.log10(stft_mag(xr, n) + 1e-5); return float((a - b).pow(2).mean(-2).sqrt().mean())

Q_A, K_A, FRAMES = 4, 256, N_AUD // 64
torch.manual_seed(SEED); cod = CodecToy(Q=Q_A, K=K_A).to(device); opt = torch.optim.Adam(cod.parameters(), lr=1e-3)
PASOS_A = 400 if FAST_DEV_RUN else 4000; g = torch.Generator().manual_seed(SEED); t0 = time.time(); hist_a = []
for s in range(PASOS_A):
    cod.train(); x = A_TR[torch.randint(0, len(A_TR), (16,), generator=g).to(device)]; nq = int(torch.randint(1, Q_A + 1, (1,), generator=g))   # quantizer dropout
    xr, ql, _ = cod(x, reinicia=(s % 50 == 0 and s > 0), n_q=nq); loss = perdida_audio(x, xr) + ql; opt.zero_grad(); loss.backward(); opt.step()
    if s % 25 == 0: hist_a.append((s, loss.item()))
print(f"códec toy: {sum(p.numel() for p in cod.parameters())/1e6:.2f}M params | {time.time()-t0:.0f}s | {FRAMES} frames/s = {SR//64} Hz | pérdida {hist_a[0][1]:.2f} -> {hist_a[-1][1]:.2f}")

In [ ]:
cod.eval(); RA = {}
with torch.no_grad():
    for nq in range(1, Q_A + 1):
        xr, _, idx = cod(A_TE, n_q=nq); RA[nq] = {"snr": snr_db(A_TE, xr), "lsd": lsd_db(A_TE, xr), "kbps": (SR // 64) * nq * math.log2(K_A) / 1000, "xr": xr, "idx": idx}
def mulaw_snr(x, bits, mu=255.0):
    y = torch.sign(x) * torch.log1p(mu * x.abs()) / math.log1p(mu); q = torch.round((y + 1) / 2 * (2 ** bits - 1)) / (2 ** bits - 1) * 2 - 1
    return snr_db(x, torch.sign(q) * ((1 + mu) ** q.abs() - 1) / mu)
MU = {b: (mulaw_snr(A_TE, b), SR * b / 1000) for b in [3, 4, 5, 6, 8]}
for nq, r in RA.items(): print(f"toy RVQ n_q={nq}: {r['kbps']:.1f} kbps  ({128/r['kbps']:.0f}x menos que PCM-16) | SNR={r['snr']:5.2f} dB | LSD={r['lsd']:5.2f} dB")
print({f"μ-law {b} bit": f"{v[1]:.0f} kbps, SNR={v[0]:.1f} dB" for b, v in MU.items()})

fig, axs = plt.subplots(2, 3, figsize=(14, 6)); t_ = np.arange(N_AUD) / SR; x0 = A_TE[0, 0].cpu().numpy()
for j, (nom, sig) in enumerate([("original", x0), ("toy codec, 1 nivel (1 kbps)", RA[1]["xr"][0, 0].cpu().numpy()), ("toy codec, 4 niveles (4 kbps)", RA[4]["xr"][0, 0].cpu().numpy())]):
    axs[0, j].plot(t_[:1500], sig[:1500], color=AZUL, lw=.7); axs[0, j].set_title(nom, fontsize=9); axs[0, j].set_ylim(-1, 1)
    axs[1, j].specgram(sig, NFFT=256, Fs=SR, noverlap=192, cmap="magma", vmin=-110); axs[1, j].set_xlabel("tiempo (s)")
axs[1, 0].set_ylabel("frecuencia (Hz)"); plt.tight_layout(); plt.show()

fig, axs = plt.subplots(1, 2, figsize=(12, 3.8))
axs[0].plot([RA[q]["kbps"] for q in RA], [RA[q]["snr"] for q in RA], "o-", color=AZUL, lw=2, label="códec toy (RVQ, 125 Hz x Q x 8 bit)")
axs[0].plot([v[1] for v in MU.values()], [v[0] for v in MU.values()], "s--", color=NARANJA, lw=2, label="μ-law (b bits/muestra)")
axs[0].set_xscale("log"); axs[0].set_xlabel("tasa (kbps, escala log)"); axs[0].set_ylabel("SNR (dB)"); axs[0].set_title("Compresión vs fidelidad en forma de onda"); axs[0].legend(fontsize=8)
axs[1].plot([RA[q]["kbps"] for q in RA], [RA[q]["lsd"] for q in RA], "o-", color=VERDE, lw=2); axs[1].set_xlabel("tasa (kbps)"); axs[1].set_ylabel("distancia log-espectral (dB, menor = mejor)")
axs[1].set_title("Fidelidad espectral del códec toy"); plt.tight_layout(); plt.show()

> **👀 Qué debes observar**
>
> * **Espectrogramas:** con 1 nivel (1 kbps) se conserva el **contorno grueso** (tono/envolvente) y se pierde el detalle armónico; con 4 niveles (4 kbps) reaparece **parte** del detalle armónico. Eso es el «refinamiento residual» en acción.
> * **SNR vs tasa:** cada nivel extra suma ~1 kbps y mejora el SNR, con rendimientos decrecientes (en la corrida de referencia con voz real FSDD: 1.5 → 2.9 dB de 1 a 4 niveles; el respaldo sintético, más regular, llega a ≈ 10 dB). **μ-law** (cuantización escalar por muestra) necesita **decenas de kbps** para lograr SNRs parecidos: un códec neuronal aprende una representación mucho más compacta *para este tipo de señal*.
> * ⚠️ **Matiz honesto:** el SNR en forma de onda **no es la métrica perceptual** que usan los códecs reales (MUSHRA, ViSQOL, PESQ): un códec puede tener SNR bajo y sonar bien porque conserva el espectro y no la fase. Y nuestro banco es diminuto (unos minutos de dígitos hablados): **no extrapoles** el SNR a EnCodec/DAC.

In [ ]:
# ¿Qué "dicen" los tokens del códec toy? Mapa de códigos (niveles x tiempo) para un clip y su perplejidad por nivel
idx = RA[4]["idx"][0].cpu().numpy()                                          # [Q, frames]
fig, axs = plt.subplots(1, 2, figsize=(13, 3.2), gridspec_kw={"width_ratios": [3, 1]})
axs[0].imshow(idx, aspect="auto", cmap="tab20", interpolation="nearest"); axs[0].set_xlabel("frame (125 por segundo)"); axs[0].set_ylabel("nivel RVQ")
axs[0].set_yticks(range(Q_A)); axs[0].set_title("Tokens del códec toy: 4 niveles x 125 frames (cada celda = un id de 0 a 255)", fontsize=9); axs[0].grid(False)
pp = [perplexity(torch.from_numpy(idx[q]), K_A) for q in range(Q_A)]
axs[1].bar([f"nivel {q+1}" for q in range(Q_A)], pp, color=AZUL); axs[1].set_ylabel("perplejidad (de 256)"); axs[1].set_title("Uso del codebook por nivel", fontsize=9)
plt.tight_layout(); plt.show()

> **👀 Qué debes observar**
>
> Cada columna es un **frame** (8 ms) y cada fila un **nivel**. Observa que el nivel 1 cambia más «lentamente» con la envolvente de las vocales, mientras los niveles posteriores parecen más ruidosos (detalle fino). La **perplejidad por nivel** te dice si el codebook de cada nivel está usado o colapsado.

> ### 🎬 GEMINI-VIDEO · ID: `M08-VID-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **Cuantización vectorial residual: cada nivel refina el error**
>
> **Qué muestra:** Animación 2D de un vector que se aproxima por pasos: el nivel 1 elige el centroide más cercano, el nivel 2 cuantiza el residuo, etc., y el error baja.
> **Formato:** MP4 16:9, 1080p | Duración: 24 s | **Texto alternativo:** Animación de cuantización vectorial residual con tres niveles que reducen el error de aproximación de un vector 2D
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Video explicativo técnico de 24 segundos, 16:9, 1080p, ritmo calmado, sin voz en off. Estilo: diagrama limpio sobre fondo blanco, tipografía sans-serif legible, paleta azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Sin ilustraciones decorativas.
> Escena fija: un plano cartesiano 2D con un punto objetivo azul etiquetado "z (vector a codificar)" en la posición aproximada (3.2, 2.1). Nota fija abajo: "Ilustración conceptual con K = 4 centroides por nivel".
> Guion por escenas:
> 0–4 s: título "RVQ: cada nivel corrige lo que dejó el anterior". Aparece el punto z y 4 puntos grises etiquetados "codebook 1" en (0,0), (4,0), (0,4), (4,4).
> 4–10 s: una línea discontinua une z con el centroide más cercano (4,4); ese centroide se vuelve naranja, etiqueta "nivel 1: q₁". Se dibuja una flecha roja desde q₁ hasta z, etiquetada "residuo r₁ = z − q₁ (error aún grande)". Contador "error: alto".
> 10–17 s: el plano se re-centra en el residuo r₁ (se traslada para que q₁ quede en el origen); aparecen 4 centroides pequeños verdes etiquetados "codebook 2" alrededor del origen. Se elige el más cercano (verde brillante, etiqueta "nivel 2: q₂"); la flecha roja se acorta. Contador "error: medio".
> 17–22 s: se repite una vez más con "codebook 3" (centroides diminutos); la flecha roja casi desaparece. Contador "error: bajo". Aparece la fórmula "ẑ = q₁ + q₂ + q₃".
> 22–24 s: texto final "Cada nivel = log₂ K bits; más niveles = más fidelidad". Se desvanece.
> Texto en pantalla en español.
> Evita: voz en off, texto ilegible, números distintos de los indicados, cambios bruscos de cámara, logotipos.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M08-VID-01.mp4` en la carpeta del módulo y reemplaza esta celda por: `<video src="assets/M08-VID-01.mp4" controls width="100%"></video>`
>
> **Mientras tanto (placeholder):**
> ```text
> nivel 1: z=(3.2,2.1) -> centroide (4,4)  residuo r1 grande
> nivel 2: r1 recentrado -> centroide verde q2     residuo r2 medio
> nivel 3: r2 -> q3                                residuo r3 pequeño   ẑ = q1+q2+q3
> ```

> **👀 Qué debes observar**
>
> El error baja **con cada nivel** y cada nivel cuesta lo mismo ($\log_2 K$ bits): es una **codificación progresiva**. Truncar niveles = bajar la tasa sin reentrenar, siempre que se haya entrenado con *quantizer dropout*.

### 10.2 EnCodec real con Hugging Face

`transformers` incluye `EncodecModel` con los pesos oficiales (`facebook/encodec_24khz`). Si no hay acceso al hub, la celda instancia el modelo con **pesos aleatorios** solo para mostrar las **formas** y la **tasa de tokens** (el audio decodificado sería ruido).

In [ ]:
# EnCodec real (Défossez et al., 2022) con Hugging Face: pesos oficiales 'facebook/encodec_24khz' (requiere acceso al hub).
try:
    from transformers import EncodecModel
    try:
        enc_m = EncodecModel.from_pretrained("facebook/encodec_24khz").eval(); real = True
    except Exception as e:
        from transformers import EncodecConfig
        enc_m = EncodecModel(EncodecConfig()).eval(); real = False            # pesos ALEATORIOS: solo para ver formas y tasas de tokens
        print("[aviso] sin acceso al hub: EnCodec con pesos aleatorios (las formas y la tasa de tokens son reales; el audio decodificado NO).")
    try:
        import torchaudio                                                     # opcional: si falta, remuestreo polifásico de SciPy
        x24 = torchaudio.functional.resample(A_TE[:1].cpu(), SR, 24000)
    except ImportError:
        from scipy.signal import resample_poly
        x24 = torch.from_numpy(resample_poly(A_TE[:1].cpu().numpy(), 24000, SR, axis=-1).astype(np.float32))
    with torch.no_grad():
        out = enc_m.encode(x24, bandwidth=6.0)
    codes = out.audio_codes                                                   # [chunks, B, n_q, T]
    print("audio_codes:", tuple(codes.shape), "-> n_q =", codes.shape[2], "| frames por segundo =", round(codes.shape[-1] * (codes.shape[0]) / 1.0, 1))
    print(f"tokens por segundo (aplanado) = {75 * codes.shape[2]} | tasa = {75 * codes.shape[2] * 10 / 1000} kbps (1024 entradas = 10 bits)  | pesos reales: {real}")
except Exception as e:
    print("transformers/torchaudio no disponibles:", type(e).__name__, str(e)[:100])

> **👀 Qué debes observar**
>
> `audio_codes` tiene forma `[chunks, batch, n_q, frames]`: con `bandwidth=6.0` hay **8 niveles** a **75 frames/s** = **600 tokens/s**, que es la cifra de nuestra tabla de presupuesto. Cámbialo a `1.5` y verás 2 niveles (150 tokens/s).

## 11. Tokenizers de vídeo

El vídeo añade el eje temporal: un **tokenizer 3D** comprime con factores $c_t\times c_s\times c_s$ (tiempo x alto x ancho), de modo que $N\approx \tfrac{T}{c_t}\cdot\tfrac{H}{c_s}\cdot\tfrac{W}{c_s}$. **Cosmos** (NVIDIA, 2025) publica tokenizers *continuos* y *discretos* (FSQ, 64 000 códigos) con compresiones espacio-temporales de **4x8x8, 8x8x8 y 8x16x16** (y 8x8 / 16x16 para imágenes); **MAGVIT-v2** (Google/CMU) usa *lookup-free quantization* para un vocabulario común de imagen y vídeo. La compresión más agresiva ahorra contexto pero **degrada la calidad** (el paper de Cosmos muestra el compromiso PSNR-compresión).

In [ ]:
# Costo de contexto de vídeo con un tokenizer 3D (compresión temporal ct x espacial cs x cs)
def tokens_clip(seg, fps, h, w, ct, cs): return -(-int(seg * fps) // ct) * (h // cs) * (w // cs)
configs = {"4x8x8": (4, 8), "8x8x8": (8, 8), "8x16x16": (8, 16)}
dur = np.array([1, 2, 5, 10, 30])
fig, ax = plt.subplots(figsize=(8, 3.8))
for (nom, (ct, cs)), col in zip(configs.items(), [AZUL, NARANJA, VERDE]):
    ax.plot(dur, [tokens_clip(d, 24, 720, 1280, ct, cs) for d in dur], "o-", color=col, label=f"Cosmos DV{nom}")
ax.axhline(128_000, color="crimson", ls="--", lw=1, label="128K tokens"); ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("duración del clip (s), 24 fps, 720p"); ax.set_ylabel("tokens (aprox.)"); ax.set_title("Vídeo: la compresión espacio-temporal decide el contexto"); ax.legend(fontsize=8); plt.show()
for nom, (ct, cs) in configs.items(): print(f"DV{nom}: clip de 5 s -> {tokens_clip(5, 24, 720, 1280, ct, cs):,} tokens")

> **👀 Qué debes observar**
>
> En escala log-log, las tres rectas son **paralelas** (el factor de compresión es constante) y separadas **órdenes de magnitud**: un clip de 30 s a 720p con 4x8x8 supera de lejos una ventana de 128K tokens, y solo las compresiones más agresivas caben. Es el análogo del «impuesto» por idioma: **el tokenizer decide cuánto contexto consume una modalidad**.

## 12. Puente con embeddings multimodales (módulo 16)

No lo desarrollamos aquí, solo lo ubicamos: los **parches ViT** (sección 6) son la entrada de CLIP/SigLIP, y un **embedding de imagen** para retrieval es el vector resultante de procesar esos tokens y **agruparlos** (pooling) o conservarlos como **multi-vector** (ColPali, módulo 16). Los **tokens discretos** de este módulo sirven sobre todo para **generar o comprimir**, no para comparar: para buscar PDFs en Políglota querrás vectores **continuos** alineados con texto. (La excepción son los *semantic IDs* de recomendación generativa, que usan RQ-VAE para indexar.)

## 13. 🏭 En producción

* **Imagen.** Los VLM abiertos más usados leen la imagen con un **encoder ViT continuo** y un proyector hacia el LLM (LLaVA); los modelos de **generación** tipo Chameleon/Emu3 usan **tokenizers discretos**; las líneas de investigación Transfusion/MAR/Fluid reducen o eliminan la cuantización. Los laboratorios de frontera cerrados no documentan sus tokenizers multimodales.
* **Audio.** **EnCodec** es el tokenizer de **MusicGen** (Meta; 50 Hz a 32 kHz según su paper) y de otros LMs de audio; **Mimi** (12.5 Hz, 1.1 kbps) es la pieza de **Moshi**, un modelo de diálogo en *streaming* (Kyutai). DAC (Descript) es un códec abierto de alta fidelidad.
* **Vídeo.** NVIDIA publica los tokenizers Cosmos (discretos con FSQ y continuos). Los costes de contexto se gestionan con compresión agresiva, jerarquías y *streaming*.
* **Costes y trade-offs.** (1) Más compresión = menos contexto pero menos calidad; (2) más niveles RVQ = más tokens por segundo; (3) los tokens de audio **aplanados** multiplican el contexto por el nº de niveles (por eso Moshi usa un transformer de profundidad); (4) un tokenizer *no se cambia* sin reentrenar el LM (como en el módulo 07, pero sin tokenizer transfer fácil).
* **Políglota.** En esta fase el valor está en **entender el coste**: indexar audio/imagen con tokens discretos en un LLM generativo sería caro; el camino del módulo 16 (embeddings continuos y multi-vector) encaja mejor con retrieval.

## 14. 🧠 Secretos de la élite

1. **Mide la perplejidad de uso del codebook, siempre.** Un VQ con $K=8192$ y perplejidad 300 es un VQ de ~300 códigos; la calidad se estanca y no sabes por qué. Es la métrica de salud nº 1.
2. **FSQ simplifica el entrenamiento:** sin codebook, sin *commitment*, sin reinicios; el paper de FSQ reporta uso de codebook casi completo y calidad comparable a VQ (vigila la **escala de entrada** al `tanh`: si es pequeña, todo redondea al mismo nivel). Es el motivo de que Cosmos use FSQ (niveles 8,8,8,5,5,5).
3. **El *quantizer dropout* es el truco de SoundStream** para tener **un modelo, muchas tasas**; sin él, recortar niveles en inferencia da basura (el decoder nunca vio esa configuración).
4. **El primer nivel de RVQ concentra la «semántica».** Mimi lo aprovecha **destilando** WavLM en el primer cuantizador (split RVQ) para que los primeros tokens lleven contenido y los demás acústica.
5. **La tasa de frames manda el coste.** 75 Hz (EnCodec) vs 12.5 Hz (Mimi) son 6x de diferencia en pasos de LM a igual nº de niveles; baja la frecuencia con un encoder más profundo y *una buena pérdida*.
6. **MSE/SNR/PSNR no son calidad perceptual.** Los códecs reales usan pérdidas **perceptuales/adversariales** y se evalúan con tests de escucha (MUSHRA) o métricas como ViSQOL; los VQGAN, con LPIPS/FID. Reportar solo PSNR engaña.
7. **Los tokenizers 1D rompen la regla «una cuadrícula = una imagen»:** TiTok representa una imagen con 32 tokens (frente a 256 de VQGAN en el mismo experimento del paper); cuando el contexto es el cuello de botella, el tokenizer es una palanca mayor que el tamaño del LM.
8. **Lo que no puede cuantizarse bien, no se cuantiza:** Transfusion y MAR defienden que, para generación de imagen, mantener **tokens continuos** (con difusión por token) evita el techo de calidad de los codebooks.

## 15. ⚠️ Errores comunes

* **Olvidar el *stop-gradient*/STE:** sin STE el encoder no entrena; sin `detach` en el codebook loss, ambos términos se pisan.
* **Inicializar el codebook al azar con escala distinta a la del encoder:** los códigos nunca ganan y quedan muertos. Inicializa **con salidas del encoder** y reinicia los muertos.
* **Reportar solo PSNR** (o SNR) para comparar tokenizers/códecs: añade una métrica perceptual y escucha/mira las muestras.
* **Comparar bitrates sin contar los niveles aplanados:** «1.1 kbps» y «600 tokens/s» no son lo mismo que «tokens por segundo que ve el LM».
* **Evaluar en el *train*:** mide en datos no vistos (test), y con imágenes/audio de **otra distribución** si vas a generalizar.
* **Pensar que discreto = mejor:** depende de la tarea (generación vs retrieval vs comprensión).

## 16. Autoevaluación

<details>
<summary>1 · (Cálculo) Chameleon tokeniza 512x512 en 1 024 tokens con un codebook de 8 192. ¿Cuántos bits por imagen, cuántos bits por píxel y cuánta compresión frente a RGB de 24 bpp?</summary>

Bits = 1 024 · log₂ 8192 = 1 024 · 13 = **13 312 bits** (≈ 1.66 KB). Píxeles = 512² = 262 144 → **0.0508 bpp**. Compresión = 24 / 0.0508 ≈ **472x** (con pérdida).

</details>

<details>
<summary>2 · (Cálculo) EnCodec a 24 kHz con 8 codebooks de 1 024 entradas y 75 frames/s: ¿tasa en kbps y tokens por minuto? ¿Y Mimi (12.5 Hz, 8 x 2048)?</summary>

EnCodec: 75·8·10 = 6 000 bit/s = **6 kbps**; tokens/min = 75·8·60 = **36 000**. Mimi: 12.5·8·11 = **1 100 bit/s = 1.1 kbps**; tokens/min = 12.5·8·60 = **6 000** (o solo 750 pasos de tiempo si el LM modela los 8 niveles con un transformer de profundidad).

</details>

<details>
<summary>3 · (Diagnóstico) Entrenas un VQ-VAE con K = 1 024 y la perplejidad de uso es 12. La reconstrucción es borrosa. ¿Qué pasa y qué pruebas, en orden?</summary>

**Colapso del codebook**: se usan ~12 códigos. Pruebas: (1) inicializar el codebook con salidas del encoder (k-means); (2) reiniciar códigos muertos periódicamente; (3) reducir la dimensión del código $d$ o usar EMA; (4) pasar a FSQ/LFQ. Verifica con la perplejidad tras cada cambio.

</details>

<details>
<summary>4 · (Diagnóstico) Tu códec RVQ entrenado con Q = 8 suena bien con 8 niveles pero con 2 niveles es ruido. ¿Por qué?</summary>

No se entrenó con *quantizer dropout*: el decoder nunca vio reconstrucciones desde pocos niveles y los niveles 1-2 no están optimizados para ser suficientes por sí solos. Solución: muestrear $n_q$ aleatorio por lote durante el entrenamiento (SoundStream).

</details>

<details>
<summary>5 · (Transferencia) Políglota quiere buscar en PDFs con diagramas (módulo 16). ¿Elegirías tokens discretos tipo VQ o parches ViT continuos? ¿Por qué?</summary>

Parches ViT **continuos** (y sus embeddings, p. ej. multi-vector): para *comparar* y buscar necesitas vectores alineados con el texto, no ids con pérdida de cuantización. Los tokens discretos sirven para **generar o comprimir**.

</details>

<details>
<summary>6 · (Transferencia) Un cliente quiere meter 1 hora de llamadas de soporte en un LLM de 128K de contexto. ¿Es viable con EnCodec a 6 kbps? ¿Y con Mimi? ¿Qué harías?</summary>

EnCodec: 2.16 M de tokens → no cabe (128K ≈ 3.6 min). Mimi aplanado: 360 K tokens → tampoco; con un transformer de profundidad sobre 12.5 Hz×3600 = 45 000 pasos sí cabría en ~45K posiciones. Alternativas: **transcribir** (ASR) y usar texto (~13 K tokens/h), resumir por tramos, o indexar con embeddings por ventanas y recuperar.

</details>

<details>
<summary>7 · (Recuerdo) ¿Para qué sirve el *straight-through estimator* en un VQ-VAE?</summary>

El argmin al codebook no es diferenciable. El STE hace $z_q = z_e + \operatorname{sg}(e - z_e)$: en forward vale $e$ y en backward copia el gradiente de $z_q$ a $z_e$, de modo que el encoder aprende de la pérdida de reconstrucción.

</details>

## 17. Referencias

Verificadas (título/autores/fecha) en arXiv en la fecha de redacción.

* Dosovitskiy et al. **An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale.** arXiv:2010.11929.
* van den Oord, Vinyals, Kavukcuoglu. **Neural Discrete Representation Learning** (VQ-VAE). arXiv:1711.00937. · Razavi et al. **Generating Diverse High-Fidelity Images with VQ-VAE-2.** arXiv:1906.00446.
* Esser, Rombach, Ommer. **Taming Transformers for High-Resolution Image Synthesis** (VQGAN). arXiv:2012.09841.
* Mentzer, Minnen, Agustsson, Tschannen. **Finite Scalar Quantization: VQ-VAE Made Simple.** arXiv:2309.15505.
* Yu et al. **Language Model Beats Diffusion — Tokenizer is Key to Visual Generation** (MAGVIT-v2, LFQ). ICLR 2024, arXiv:2310.05737.
* Chameleon Team. **Chameleon: Mixed-Modal Early-Fusion Foundation Models.** arXiv:2405.09818. · Wang et al. **Emu3: Next-Token Prediction is All You Need.** arXiv:2409.18869.
* Zhou et al. **Transfusion: Predict the Next Token and Diffuse Images with One Multi-Modal Model.** arXiv:2408.11039. · Li et al. **Autoregressive Image Generation without Vector Quantization.** arXiv:2406.11838. · Fan et al. **Fluid: Scaling Autoregressive Text-to-image Generative Models with Continuous Tokens.** arXiv:2410.13863.
* Yu et al. **An Image is Worth 32 Tokens for Reconstruction and Generation** (TiTok). arXiv:2406.07550. · Bachmann et al. **FlexTok: Resampling Images into 1D Token Sequences of Flexible Length.** arXiv:2502.13967.
* Zeghidour et al. **SoundStream: An End-to-End Neural Audio Codec.** arXiv:2107.03312. · Défossez et al. **High Fidelity Neural Audio Compression** (EnCodec). arXiv:2210.13438. · Kumar et al. **High-Fidelity Audio Compression with Improved RVQGAN** (DAC). arXiv:2306.06546.
* Défossez et al. **Moshi: a speech-text foundation model for real-time dialogue** (Mimi). arXiv:2410.00037. · Zhang et al. **SpeechTokenizer: Unified Speech Tokenizer for Speech Large Language Models.** arXiv:2308.16692.
* Copet et al. **Simple and Controllable Music Generation** (MusicGen). arXiv:2306.05284.
* NVIDIA. **Cosmos World Foundation Model Platform for Physical AI.** arXiv:2501.03575.
* Liu et al. **Visual Instruction Tuning** (LLaVA). arXiv:2304.08485. · Rajput et al. **Recommender Systems with Generative Retrieval.** arXiv:2305.05065.
* Documentación: `torchvision.datasets.MNIST`, `transformers.EncodecModel`, `scikit-learn` (`load_sample_image`).

## 18. Siguiente paso

El **proyecto** (`08_proyecto_tokens_multimodales_audio.ipynb`) te pide tokenizar **una foto real y un clip de audio** y medir **reconstrucción y compresión** frente a JPEG y μ-law. El **módulo 09** vuelve al texto: la **capa de embedding** de los LLM, donde cada id que hemos aprendido a construir se convierte en un vector.